In [5]:
import os, re, json, requests
from dotenv import load_dotenv

load_dotenv()
LAW_API = os.getenv("LAW_API")

LAW_MAP_PATH = "../data/law_map.jsonl"
NEED_LAWS_PATH = "../data/need_laws.json"
RAW_DIR = "../data/law_rawdata/"
os.makedirs(RAW_DIR, exist_ok=True)

def get_url(law_id: str) -> str:
    return f"https://www.law.go.kr/DRF/lawService.do?OC={LAW_API}&target=eflaw&ID={law_id}&type=xml"

def sanitize(name: str) -> str:
    return re.sub(r"[^0-9A-Za-z가-힣]+", "_", name).strip("_")

def load_jsonl(path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def load_need_laws(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise ValueError("need_laws.json must be a JSON object: { '법령명': '파일명', ... }")
    return data

# 1) 입력 로드
need_laws = load_need_laws(NEED_LAWS_PATH)   # {"민법": "Civil_Law", ...}
need_names = set(need_laws.keys())

law_map = load_jsonl(LAW_MAP_PATH)

# 2) law_name / law_short_name -> law_id 인덱스 구성
name_to_id: dict[str, str] = {}
short_to_id: dict[str, str] = {}

for row in law_map:
    law_id = (row.get("law_id") or "").strip()
    if not law_id:
        continue

    name = (row.get("law_name") or "").strip()
    short = (row.get("law_short_name") or "").strip()

    # 먼저 들어온 값을 유지 (중복 row가 있어도 안정적으로)
    if name and name not in name_to_id:
        name_to_id[name] = law_id
    if short and short not in short_to_id:
        short_to_id[short] = law_id

# 3) 필요한 법들에 대해 id 찾고 저장
missing = []
saved = 0

for need_name, out_base in need_laws.items():
    law_id = name_to_id.get(need_name) or short_to_id.get(need_name)

    if not law_id:
        missing.append(need_name)
        continue

    url = get_url(law_id)
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()

    fname = sanitize(out_base)
    out_path = os.path.join(RAW_DIR, f"{fname}.xml")
    with open(out_path, "wb") as f:
        f.write(resp.content)

    print(f"{need_name} ({law_id}) -> {fname}.xml saved")
    saved += 1

print(f"\nDone. saved={saved}, missing={len(missing)}")
if missing:
    print("Missing in law_map.jsonl:")
    for n in missing:
        print(f"- {n}")


소비자기본법 (001589) -> Consumer_Basic_Law.xml saved
전자상거래 등에서의 소비자보호에 관한 법률 (009318) -> E_Commerce_Consumer_Law.xml saved
전자문서 및 전자거래 기본법 (002000) -> E_Transaction_Law.xml saved
콘텐츠산업 진흥법 (009280) -> Content_Industry_Promotion_Law.xml saved
제조물 책임법 (002039) -> Product_Liability_Law.xml saved
약관의 규제에 관한 법률 (000667) -> Terms_Regulation_Law.xml saved
표시ㆍ광고의 공정화에 관한 법률 (002011) -> Fair_Ads_Law.xml saved
할부거래에 관한 법률 (000355) -> Installment_Sales_Law.xml saved
방문판매 등에 관한 법률 (000354) -> Direct_Sales_Law.xml saved
민법 (001706) -> Civil_Law.xml saved
상법 (001702) -> Commercial_Law.xml saved

Done. saved=11, missing=0
